# PCCP断丝信号特征挖掘

本 notebook 对应开发方案中的 Notebook01-Notebook08，并新增 Notebook09 分类测试。输入是六类已经计算好的特征向量，不重新计算原始波形特征；目标是从数百个候选特征中筛出能够稳定表征 PCCP 断丝响应的少量核心特征。全流程核心代码位于 `src/pccp_feature_mining`，notebook 负责组织实验、展示结果、解释指标含义并给出结论。

## 00 环境初始化与显示工具

本节只做三件事：定位项目根目录、导入显示工具、定义统一的表格/图片展示函数。后续每个表都用 `show_table()` 输出“表号 + 标题 + 表格”，每张结果图都用 `show_image()` 在 notebook 内直接显示。

In [ ]:
from pathlib import Path
import sys
import json
import pandas as pd
from IPython.display import Image, Markdown, display


def find_project_root(start: Path) -> Path:
    """向上查找.git，保证从notebooks目录启动时也能定位项目根目录。"""
    p = start.resolve()
    for candidate in [p] + list(p.parents):
        if (candidate / '.git').exists():
            return candidate
    return p


PROJECT_ROOT = find_project_root(Path.cwd())
SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from pccp_feature_mining.feature_schema import add_feature_meaning_columns, describe_feature_list

RUN_DIR = PROJECT_ROOT / 'outputs/PCCP_feature_mining_notebook/manual_run'
RUN_DIR.mkdir(parents=True, exist_ok=True)

TABLE_NO = 0
FIGURE_NO = 0


def show_table(title: str, frame: pd.DataFrame, rows: int | None = None) -> pd.DataFrame:
    """显示带编号标题的表格，并自动补充特征中文含义列。"""
    global TABLE_NO
    TABLE_NO += 1
    out = add_feature_meaning_columns(frame)
    if rows is not None:
        out = out.head(rows)
    display(Markdown(f'**表 {TABLE_NO} {title}**'))
    display(out)
    return out


def show_image(title: str, path: Path) -> None:
    """显示带编号标题的结果图。"""
    global FIGURE_NO
    FIGURE_NO += 1
    display(Markdown(f'**图 {FIGURE_NO} {title}**'))
    display(Image(filename=str(path)))


print('项目根目录:', PROJECT_ROOT)
print('默认输出目录:', RUN_DIR)

## 01 数据读取与质量检查

本节实现“数据读取与质量检查”。算法上先递归读取六类输入目录下的 `features*.csv`，使用显式目录标签覆盖 CSV 内部可能不一致的标签，再统一生成 `source_label`、`target_label`、`signal_family`、`flow_condition`、`event_id` 和 `sampling_group_id`。

本版新增行级有效特征检测：旧逻辑只检查 CSV 表头列数和是否为空表，因此类似 `features_20260905_173711_part_0001.csv` 这种“661列、17066行，但 6568 行没有任何有效频带特征值”的文件不会被跳过。现在读取时会计算每行非空有效特征数，并剔除低于 `CONFIG.min_valid_feature_values_per_row` 的行；这些行不会进入 Notebook02-Notebook09 的分布、判别、冗余、组合、稳定性、跨流速或分类分析。

可调参数：`min_feature_csv_columns` 控制文件级表头列数门槛；`min_valid_feature_values_per_row` 控制行级有效特征数门槛。若后续特征总数变化，建议设置为“预期有效特征数的 10%-20%”或至少 50，本 notebook 当前使用 100。

 
 ### missing_rate 缺失率公式

$$\text{missing\_rate} = 1 - \frac{\text{finite\_count}}{N}$$

其中：
- $N$ = 总行数（`len(x)`）
- $\text{finite\_count}$ = 有效值数量（去除 `inf`、`-inf`、`NaN` 后的非空值数量）

代码实现（`quality_control.py:50`）：
```python
finite = s.replace([np.inf, -np.inf], np.nan).dropna()
missing_rate = 1.0 - finite.size / n
```

含义：`missing_rate` 表示该特征中无效值（缺失/无穷大）占总样本数的比例，值域 $[0, 1]$。

In [ ]:
from pccp_feature_mining.config import FeatureInput, MiningConfig
from pccp_feature_mining.data_loader import load_feature_dataset
from pccp_feature_mining.quality_control import build_dataset_summary, build_feature_quality, feature_list_frame
from pccp_feature_mining.report_generator import write_csv, write_markdown_summary

FEATURE_INPUTS = [
    FeatureInput('FL00', PROJECT_ROOT / r'outputs/DATA09_v0-flow_features_20260904_121425/DATA09_v0-flow_features'),
    FeatureInput('FL05', PROJECT_ROOT / r'outputs/DATA09_v0-flow_features_20260904_121425/DATA09_v0.5-flow_features'),
    FeatureInput('BK00', PROJECT_ROOT / r'outputs/DATA09_multi_label_single_event_features/v0-bk_features_BK00'),
    FeatureInput('QJ00', PROJECT_ROOT / r'outputs/DATA09_multi_label_single_event_features/v0-qj_features_QJ00'),
    FeatureInput('BK05', PROJECT_ROOT / r'outputs/DATA09_multi_label_single_event_features/v05-bk_features_BK05'),
    FeatureInput('QJ05', PROJECT_ROOT / r'outputs/DATA09_multi_label_single_event_features/v05-qj_features_QJ05'),
]

CONFIG = MiningConfig(
    # 输出目录：存放所有分析结果、中间产物和可视化图表
    output_dir=PROJECT_ROOT / 'outputs/PCCP_feature_mining_notebook',
    # 特征输入列表：定义六类信号的特征CSV路径，每类信号一个目录
    feature_inputs=tuple(FEATURE_INPUTS),
    # 每个CSV最少列数：剔除列数不足的异常特征文件
    min_feature_csv_columns=100,
    # 每行最少有效特征值：剔除缺失值过多的样本行
    min_valid_feature_values_per_row=100,
    # 随机种子：保证实验可复现
    random_state=42,
    # 相关性阈值：Spearman相关系数超过此值的特征视为冗余，保留其中一个
    correlation_threshold=0.92,
    # Bootstrap重采样轮数：用于估计特征排名的稳定性
    bootstrap_rounds=200,
    # Bootstrap取前K个：每轮重采样后保留排名前K的特征，统计出现频率
    bootstrap_top_ks=(10, 20, 30),
    # mRMR保留特征数：最大相关最小冗余算法保留的特征数量
    mrmr_top_n=80,
    # 相关性分析最大样本数：超过此行数时随机抽样，加速计算
    max_correlation_rows=6000,
    # 分布分析展示Top-N：单特征判别分析中展示排名前N的特征分布图
    distribution_top_n=12,
    # 投影分析最大样本数：PCA/UMAP降维可视化时的最大样本数
    projection_max_rows=8000,
    # 组合特征数量列表：测试不同大小的特征子集（5,10,20,30,50个特征）
    combination_feature_counts=(5, 10, 20, 30, 50),
    # SFS候选特征数：序列前向选择的初始候选特征池大小
    sfs_candidate_count=30,
    # SFS最大选择数：序列前向选择算法最多选择的特征数量
    sfs_max_selected=15,
    # 分类测试特征数量列表：分类评估时测试的特征子集大小
    classification_feature_counts=(5, 10, 20, 30, 50),
    # 分类训练集每类最大样本数：超过时随机抽样，控制训练时间
    classification_max_train_rows_per_class=3000,
    # 每标签最大样本数：None表示使用全部数据；快速调试可设为1000
    max_rows_per_label=None,  # 快速调试可设为1000；正式分析保持None。
)

config_table = pd.DataFrame([
    {'参数': 'min_feature_csv_columns', '当前值': CONFIG.min_feature_csv_columns, '含义': '文件级最低列数；列数低于该值的CSV直接跳过'},
    {'参数': 'min_valid_feature_values_per_row', '当前值': CONFIG.min_valid_feature_values_per_row, '含义': '行级最低有效特征数；低于该值的样本行不进入后续分析'},
    {'参数': 'bootstrap_rounds', '当前值': CONFIG.bootstrap_rounds, '含义': 'Bootstrap重复抽样次数'},
    {'参数': 'bootstrap_top_ks', '当前值': str(CONFIG.bootstrap_top_ks), '含义': '统计Top-K出现频率的K值'},
])
show_table('Notebook01 关键质量控制参数', config_table)

In [ ]:
dataset = load_feature_dataset(
    feature_inputs=CONFIG.feature_inputs,
    min_feature_csv_columns=CONFIG.min_feature_csv_columns,
    min_valid_feature_values_per_row=CONFIG.min_valid_feature_values_per_row,
    max_rows_per_label=CONFIG.max_rows_per_label,
    random_state=CONFIG.random_state,
)
df = dataset.frame
feature_cols = list(dataset.feature_columns)

dataset_summary = build_dataset_summary(df)
feature_quality = build_feature_quality(df, feature_cols)
feature_list = feature_list_frame(feature_cols)
load_issues = dataset.load_report[
    (dataset.load_report['status'] != 'ok') | (dataset.load_report['dropped_low_valid_feature_rows'] > 0)
].copy()

write_csv(dataset.load_report, RUN_DIR / 'load_report.csv')
write_csv(dataset_summary, RUN_DIR / 'dataset_summary.csv')
write_csv(feature_list, RUN_DIR / 'feature_list.csv')
write_csv(feature_quality, RUN_DIR / 'quality_report.csv')

print(f'样本行数: {len(df):,}, 特征数: {len(feature_cols):,}')
show_table('六类样本读取汇总', dataset_summary)
show_table('有效特征清单样例', feature_list, rows=10)
show_table('特征质量报告Top20', feature_quality, rows=10)
show_table('读取异常与低有效特征行审计', load_issues)

## 02 特征分布分析

本章对候选特征在不同标签分组上的分布进行可视化分析，目的是直观判断哪些特征在不同类别之间存在明显差异。

- $可视化方法说明$

| 图表 | 英文全称 | 中文全称 | 观察目的 |
|------|----------|----------|----------|
| 箱线图 | Box Plot | 箱线图 | 观察中位数（箱体中线）和四分位距（箱体高度），判断分布中心和离散程度 |
| KDE分布图 | Kernel Density Estimation | 核密度估计图 | 观察分布重叠程度，曲线重叠越多说明特征区分能力越弱 |
| PCA投影图 | Principal Component Analysis | 主成分分析投影图 | 用二维投影观察整体特征空间的类间结构，点集分离越好说明特征区分能力越强 |
| UMAP投影图 | Uniform Manifold Approximation and Projection | 均匀流形逼近与投影 | 非线性降维投影，保留局部结构，可发现PCA无法捕捉的非线性类间差异 |

- $类间均值方差与类内方差均值的计算$

对**某个特征**，先将全部样本按标签分组（共 $k$ 类），再分别计算两种方差：

**“类”的定义**：这里的“类”指按标签（`source_label`，如 `BK00/BK05/FL00/FL05/QJ00/QJ05`）分组后的每一组信号。第 $i$ 个类 = 标签为第 $i$ 种的所有样本的集合。例如对某特征，把所有样本按 `source_label` 分成6组（用6个标签时 $k=6$），每组样本数量可能不同，但都属于同一个信号类别。

以特征 `b_40k_60k__SF` 为例：把该特征所有样本按 `source_label` 分成6组 → 每个标签算出一个组均值 $m_i$ → 这6个 $m_i$ 求方差得类间均值方差；每个标签算组内内部方差，6个求平均得类内方差均值。

**1. 类间均值方差 $\sigma^2_{between}$**（衡量各类均值之间的离散程度）

$$
\sigma^2_{between} = \frac{1}{k}\sum_{i=1}^{k}(m_i - \bar{m})^2
$$

其中 $m_i$ 是第 $i$ 类的均值（该组所有样本的特征值平均值），$\bar{m}$ 是 $k$ 个类均值的平均值。步骤：先求各类均值 `group_means`，再对这 $k$ 个均值求总体方差（`var(ddof=0)`，除以 $k$）。

**2. 类内方差均值 $\sigma^2_{within}$**（衡量各类内部波动的平均程度）

$$
\sigma^2_{within} = \frac{1}{k}\sum_{i=1}^{k}s_i^2
$$

其中 $s_i^2$ 是第 $i$ 类内部的方差（该组样本内部的离散程度）。步骤：先算每类内部方差（`groupby(labels).var(ddof=0)`），再对这 $k$ 个类内方差求平均（`.mean()`）。

**3. 二者比值（F-ratio）**

$$
\text{f\_ratio} = \frac{\sigma^2_{between}}{\sigma^2_{within} + \epsilon}, \quad \epsilon = 10^{-12}
$$

$\epsilon$ 用于防止分母为零。该值越大，说明特征在各类之间均值差异越明显（类间越分散、类内越集中），越适合用于分布展示。

**本章子节结构**

- **2.1** 六标签分布比较：`BK00/BK05/FL00/FL05/QJ00/QJ05`
- **2.2** `BK00 vs BK05`：同为断丝信号，比较零流速与半流速的差异
- **2.3** `BK00 vs QJ00`：零流速下，断丝与敲击的差异
- **2.4** `BK05 vs QJ05`：半流速下，断丝与敲击的差异
- **2.5** `(BK00+BK05) vs (QJ00+QJ05)`：断丝集合 vs 敲击集合
- **2.6** `(BK00+BK05) vs (QJ00+QJ05+FL00+FL05)`：断丝集合 vs 全部非断丝集合

### 2.1 六标签分布比较

比较 `BK00/BK05/FL00/FL05/QJ00/QJ05` 六个来源标签上的特征分布。

In [ ]:
from pccp_feature_mining.distribution_analysis import (
    select_distribution_features,
    run_pca_projection,
    run_umap_projection,
    plot_feature_kde_by_label,
    plot_projection,
)
from pccp_feature_mining.visualization import plot_final_ranking, plot_top_feature_boxplots

six_class_distribution_features = select_distribution_features(df, feature_cols, top_n=CONFIG.distribution_top_n)
pca_projection, pca_explained = run_pca_projection(df, feature_cols, max_rows=CONFIG.projection_max_rows, random_state=CONFIG.random_state)
umap_projection = run_umap_projection(df, feature_cols, max_rows=min(CONFIG.projection_max_rows, 5000), random_state=CONFIG.random_state)

write_csv(add_feature_meaning_columns(six_class_distribution_features), RUN_DIR / 'six_class_distribution_features.csv')
write_csv(pca_projection, RUN_DIR / 'six_class_pca_projection.csv')
write_csv(pca_explained, RUN_DIR / 'six_class_pca_explained_variance.csv')
write_csv(umap_projection, RUN_DIR / 'six_class_umap_projection.csv')

boxplot_path = RUN_DIR / 'plots/six_class_feature_boxplots.png'
kde_path = RUN_DIR / 'plots/six_class_feature_kde.png'
pca_path = RUN_DIR / 'plots/six_class_pca.png'
umap_path = RUN_DIR / 'plots/six_class_umap.png'

plot_top_feature_boxplots(df, six_class_distribution_features['feature'].head(12).tolist(), boxplot_path, max_features=12, log_scale=True)
plot_feature_kde_by_label(df, six_class_distribution_features['feature'].head(8).tolist(), kde_path)
plot_projection(pca_projection, 'PC1', 'PC2', pca_path, '六类特征PCA投影')
if not umap_projection.empty:
    plot_projection(umap_projection, 'UMAP1', 'UMAP2', umap_path, '六类特征UMAP投影')

show_table('六类分布差异最大的展示特征', six_class_distribution_features)
show_table('PCA解释方差', pca_explained)
show_image('六类特征箱线图', boxplot_path)
show_image('六类特征KDE分布图', kde_path)
show_image('六类特征PCA投影图', pca_path)
if not umap_projection.empty:
    show_image('六类特征UMAP投影图', umap_path)

In [ ]:
# 2.1 已在上方完成（六类特征分布分析）
print('2.1 六标签分布比较已在上方完成')

### 2.2 BK00 vs BK05

同为断丝信号，比较零流速（BK00）与半流速（BK05）的特征分布差异。

In [ ]:
# 2.2 BK00 vs BK05
df_bk00_bk05 = df[df['source_label'].isin(['BK00', 'BK05'])].copy()

def compute_two_class_f_ratio(frame, feature_columns, label_col='source_label', class_a='BK00', class_b='BK05'):
    """计算二分类F-ratio：类间方差/类内方差均值。"""
    from pccp_feature_mining.feature_schema import impute_with_median, numeric_feature_frame
    x = impute_with_median(numeric_feature_frame(frame, feature_columns))
    rows = []
    labels = frame[label_col].astype(str)
    for feature in feature_columns:
        values = x[feature]
        group_means = values.groupby(labels).mean()
        between = float(group_means.var(ddof=0))
        within = float(values.groupby(labels).var(ddof=0).mean())
        score = between / (within + 1e-12)
        rows.append({'feature': feature, 'f_ratio': score, 'between_var': between, 'within_var': within})
    return pd.DataFrame(rows).sort_values('f_ratio', ascending=False)

bk00_bk05_f_ratio = compute_two_class_f_ratio(df_bk00_bk05, feature_cols, class_a='BK00', class_b='BK05')
top_features_2_2 = bk00_bk05_f_ratio.head(CONFIG.distribution_top_n)['feature'].tolist()

# 箱线图
boxplot_path_2_2 = RUN_DIR / 'plots/bk00_vs_bk05_boxplots.png'
plot_top_feature_boxplots(df_bk00_bk05, top_features_2_2, boxplot_path_2_2, max_features=12, log_scale=True)

# KDE图
kde_path_2_2 = RUN_DIR / 'plots/bk00_vs_bk05_kde.png'
plot_feature_kde_by_label(df_bk00_bk05, top_features_2_2[:8], kde_path_2_2)

# PCA投影
pca_projection_2_2, pca_explained_2_2 = run_pca_projection(df_bk00_bk05, feature_cols, max_rows=CONFIG.projection_max_rows, random_state=CONFIG.random_state)
pca_path_2_2 = RUN_DIR / 'plots/bk00_vs_bk05_pca.png'
plot_projection(pca_projection_2_2, 'PC1', 'PC2', pca_path_2_2, 'BK00 vs BK05 PCA投影')

show_table('2.2 BK00 vs BK05 F-ratio Top20', bk00_bk05_f_ratio.head(20))
show_image('2.2 BK00 vs BK05 箱线图', boxplot_path_2_2)
show_image('2.2 BK00 vs BK05 KDE分布图', kde_path_2_2)
show_image('2.2 BK00 vs BK05 PCA投影图', pca_path_2_2)

### 2.3 BK00 vs QJ00

零流速下，断丝信号（BK00）与敲击信号（QJ00）的特征分布差异。

In [ ]:
# 2.3 BK00 vs QJ00
df_bk00_qj00 = df[df['source_label'].isin(['BK00', 'QJ00'])].copy()

bk00_qj00_f_ratio = compute_two_class_f_ratio(df_bk00_qj00, feature_cols, class_a='BK00', class_b='QJ00')
top_features_2_3 = bk00_qj00_f_ratio.head(CONFIG.distribution_top_n)['feature'].tolist()

boxplot_path_2_3 = RUN_DIR / 'plots/bk00_vs_qj00_boxplots.png'
plot_top_feature_boxplots(df_bk00_qj00, top_features_2_3, boxplot_path_2_3, max_features=12, log_scale=True)

kde_path_2_3 = RUN_DIR / 'plots/bk00_vs_qj00_kde.png'
plot_feature_kde_by_label(df_bk00_qj00, top_features_2_3[:8], kde_path_2_3)

pca_projection_2_3, pca_explained_2_3 = run_pca_projection(df_bk00_qj00, feature_cols, max_rows=CONFIG.projection_max_rows, random_state=CONFIG.random_state)
pca_path_2_3 = RUN_DIR / 'plots/bk00_vs_qj00_pca.png'
plot_projection(pca_projection_2_3, 'PC1', 'PC2', pca_path_2_3, 'BK00 vs QJ00 PCA投影')

show_table('2.3 BK00 vs QJ00 F-ratio Top20', bk00_qj00_f_ratio.head(20))
show_image('2.3 BK00 vs QJ00 箱线图', boxplot_path_2_3)
show_image('2.3 BK00 vs QJ00 KDE分布图', kde_path_2_3)
show_image('2.3 BK00 vs QJ00 PCA投影图', pca_path_2_3)

### 2.4 BK05 vs QJ05

半流速下，断丝信号（BK05）与敲击信号（QJ05）的特征分布差异。

In [ ]:
# 2.4 BK05 vs QJ05
df_bk05_qj05 = df[df['source_label'].isin(['BK05', 'QJ05'])].copy()

bk05_qj05_f_ratio = compute_two_class_f_ratio(df_bk05_qj05, feature_cols, class_a='BK05', class_b='QJ05')
top_features_2_4 = bk05_qj05_f_ratio.head(CONFIG.distribution_top_n)['feature'].tolist()

boxplot_path_2_4 = RUN_DIR / 'plots/bk05_vs_qj05_boxplots.png'
plot_top_feature_boxplots(df_bk05_qj05, top_features_2_4, boxplot_path_2_4, max_features=12, log_scale=True)

kde_path_2_4 = RUN_DIR / 'plots/bk05_vs_qj05_kde.png'
plot_feature_kde_by_label(df_bk05_qj05, top_features_2_4[:8], kde_path_2_4)

pca_projection_2_4, pca_explained_2_4 = run_pca_projection(df_bk05_qj05, feature_cols, max_rows=CONFIG.projection_max_rows, random_state=CONFIG.random_state)
pca_path_2_4 = RUN_DIR / 'plots/bk05_vs_qj05_pca.png'
plot_projection(pca_projection_2_4, 'PC1', 'PC2', pca_path_2_4, 'BK05 vs QJ05 PCA投影')

show_table('2.4 BK05 vs QJ05 F-ratio Top20', bk05_qj05_f_ratio.head(20))
show_image('2.4 BK05 vs QJ05 箱线图', boxplot_path_2_4)
show_image('2.4 BK05 vs QJ05 KDE分布图', kde_path_2_4)
show_image('2.4 BK05 vs QJ05 PCA投影图', pca_path_2_4)

### 2.5 (BK00+BK05) vs (QJ00+QJ05)

断丝信号集合（BK00+BK05）与敲击信号集合（QJ00+QJ05）的特征分布差异。

In [ ]:
# 2.5 (BK00+BK05) vs (QJ00+QJ05)
df_bk_all = df[df['source_label'].isin(['BK00', 'BK05'])].copy()
df_bk_all['binary_label'] = 'BK'

df_qj_all = df[df['source_label'].isin(['QJ00', 'QJ05'])].copy()
df_qj_all['binary_label'] = 'QJ'

df_bk_qj = pd.concat([df_bk_all, df_qj_all], ignore_index=True)

bk_qj_f_ratio = compute_two_class_f_ratio(df_bk_qj, feature_cols, label_col='binary_label', class_a='BK', class_b='QJ')
top_features_2_5 = bk_qj_f_ratio.head(CONFIG.distribution_top_n)['feature'].tolist()

boxplot_path_2_5 = RUN_DIR / 'plots/bk_all_vs_qj_all_boxplots.png'
plot_top_feature_boxplots(df_bk_qj, top_features_2_5, boxplot_path_2_5, max_features=12, log_scale=True)

kde_path_2_5 = RUN_DIR / 'plots/bk_all_vs_qj_all_kde.png'
plot_feature_kde_by_label(df_bk_qj, top_features_2_5[:8], kde_path_2_5)

pca_projection_2_5, pca_explained_2_5 = run_pca_projection(df_bk_qj, feature_cols, max_rows=CONFIG.projection_max_rows, random_state=CONFIG.random_state)
pca_path_2_5 = RUN_DIR / 'plots/bk_all_vs_qj_all_pca.png'
plot_projection(pca_projection_2_5, 'PC1', 'PC2', pca_path_2_5, '(BK00+BK05) vs (QJ00+QJ05) PCA投影')

show_table('2.5 (BK00+BK05) vs (QJ00+QJ05) F-ratio Top20', bk_qj_f_ratio.head(20))
show_image('2.5 (BK00+BK05) vs (QJ00+QJ05) 箱线图', boxplot_path_2_5)
show_image('2.5 (BK00+BK05) vs (QJ00+QJ05) KDE分布图', kde_path_2_5)
show_image('2.5 (BK00+BK05) vs (QJ00+QJ05) PCA投影图', pca_path_2_5)

### 2.6 (BK00+BK05) vs (QJ00+QJ05+FL00+FL05)

断丝信号集合（BK00+BK05）与全部非断丝信号集合（QJ00+QJ05+FL00+FL05）的特征分布差异。

In [ ]:
# 2.6 (BK00+BK05) vs (QJ00+QJ05+FL00+FL05)
df_bk_all = df[df['source_label'].isin(['BK00', 'BK05'])].copy()
df_bk_all['binary_label'] = 'BK'

df_non_bk = df[df['source_label'].isin(['QJ00', 'QJ05', 'FL00', 'FL05'])].copy()
df_non_bk['binary_label'] = 'NONBK'

df_bk_nonbk = pd.concat([df_bk_all, df_non_bk], ignore_index=True)

bk_nonbk_f_ratio = compute_two_class_f_ratio(df_bk_nonbk, feature_cols, label_col='binary_label', class_a='BK', class_b='NONBK')
top_features_2_6 = bk_nonbk_f_ratio.head(CONFIG.distribution_top_n)['feature'].tolist()

boxplot_path_2_6 = RUN_DIR / 'plots/bk_all_vs_nonbk_all_boxplots.png'
plot_top_feature_boxplots(df_bk_nonbk, top_features_2_6, boxplot_path_2_6, max_features=12, log_scale=True)

kde_path_2_6 = RUN_DIR / 'plots/bk_all_vs_nonbk_all_kde.png'
plot_feature_kde_by_label(df_bk_nonbk, top_features_2_6[:8], kde_path_2_6)

pca_projection_2_6, pca_explained_2_6 = run_pca_projection(df_bk_nonbk, feature_cols, max_rows=CONFIG.projection_max_rows, random_state=CONFIG.random_state)
pca_path_2_6 = RUN_DIR / 'plots/bk_all_vs_nonbk_all_pca.png'
plot_projection(pca_projection_2_6, 'PC1', 'PC2', pca_path_2_6, '(BK00+BK05) vs (QJ00+QJ05+FL00+FL05) PCA投影')

show_table('2.6 (BK00+BK05) vs (QJ00+QJ05+FL00+FL05) F-ratio Top20', bk_nonbk_f_ratio.head(20))
show_image('2.6 (BK00+BK05) vs (QJ00+QJ05+FL00+FL05) 箱线图', boxplot_path_2_6)
show_image('2.6 (BK00+BK05) vs (QJ00+QJ05+FL00+FL05) KDE分布图', kde_path_2_6)
show_image('2.6 (BK00+BK05) vs (QJ00+QJ05+FL00+FL05) PCA投影图', pca_path_2_6)

## 03 单特征判别能力分析

本节对每个单特征计算二分类判别能力，回答"该特征在多大程度上能单独区分断丝与非断丝"。

**核心指标与公式**（记正类样本集 $P$、负类样本集 $N$，特征取值 $x$）

- **AUC（ROC 曲线下面积）**：衡量特征值排序能把正类排在负类前面的概率：
  $$\mathrm{AUC}=\frac{1}{|N|}\sum_{j\in N}\frac{\#\{i\in P: x_i > x_j\}}{|P|}$$
  并派生两个无量纲量：
  $$\mathrm{auc\_abs}=\max(\mathrm{AUC},\,1-\mathrm{AUC}),\qquad \mathrm{auc\_lift}=\mathrm{auc\_abs}-0.5$$
  $0.5$ 表示随机，越接近 $1$ 分离越强；取无方向是为了"哪一类偏大"不预先假设。

- **Wasserstein 距离（推土机距离 EMD）**：衡量两类分布的整体位移，
  $$W_1(P,N)=\int_{-\infty}^{+\infty}\big\lvert F_P(u)-F_N(u)\big\rvert\,\mathrm{d}u$$
  即两个经验 CDF 之间围成的面积；归一化 `wasserstein_norm = W_1 / IQR`（IQR 为合并样本的四分位距），越大越分离。

- **Cliff's delta**：衡量"正类样本取值大于负类样本"的优势概率，
  $$\delta = \frac{2U}{|P||N|}-1=\frac{\#(x_p>x_n)-\#(x_p<x_n)}{|P||N|},\qquad \delta\in[-1,1]$$
  绝对值越大，两类两两比较的优势越明显。

- **互信息（Mutual Information）**：衡量非线性依赖，
  $$I(x;y)=\sum_{x,y}p(x,y)\log\frac{p(x,y)}{p(x)p(y)}$$
  用 `mutual_info_classif`（k近邻/直方图估计）计算，值越大依赖越强。

- **综合判别分**（`discrimination_score`，用于 mRMR、冗余组和最终排序）：
  $$\mathrm{discrimination\_score}=0.55\cdot2\cdot\max(\mathrm{auc\_lift},0)+0.30\cdot\min(|\delta|,1)+0.15\cdot\frac{I(x;y)}{1+I(x;y)}$$

**本章任务划分**：按五个二级比较任务输出结果：
- 3.1 `BK00 vs NONBK00`（零流速断丝 vs 零流速背景+敲击）
- 3.2 `BK00 vs QJ00`（零流速断丝 vs 零流速敲击）
- 3.3 `BK05 vs QJ05`（半流速断丝 vs 半流速敲击）
- 3.4 `BK05 vs NONBK05`（半流速断丝 vs 半流速背景+敲击）
- 3.5 `BK vs NONBK`（两种流速全部断丝 vs 全部非断丝，最终排序主任务）

**参考文献**

[1] Fawcett T. An introduction to ROC analysis[J]. *Pattern Recognition Letters*, 2006, 27(8): 861-874.
[2] Cliff N. Dominance statistics: ordinal analyses to answer ordinal questions[J]. *Psychological Bulletin*, 1993, 114(3): 494-509.

In [ ]:
from pccp_feature_mining.feature_discrimination import evaluate_feature_discrimination

feature_discrimination = evaluate_feature_discrimination(df, feature_cols, random_state=CONFIG.random_state)
write_csv(add_feature_meaning_columns(feature_discrimination), RUN_DIR / 'feature_discrimination.csv')

metric_explain = pd.DataFrame([
    {'列名': 'auc_abs', '含义': '无方向AUC，越接近1越好，0.5表示随机', '公式': 'max(AUC, 1-AUC)'},
    {'列名': 'auc_lift', '含义': 'AUC相对随机分类的提升，越大越好', '公式': 'auc_abs - 0.5'},
    {'列名': 'wasserstein_norm', '含义': '按IQR/标准差归一化的分布距离，越大越好', '公式': 'W(pos, neg) / scale'},
    {'列名': 'abs_cliff_delta', '含义': '两类两两比较优势强度，0到1，越大越好', '公式': '|P(pos>neg)-P(pos<neg)|'},
    {'列名': 'mutual_info', '含义': '特征与类别的信息量，越大越好', '公式': 'MI(feature, label)'},
    {'列名': 'discrimination_score', '含义': '单特征综合判别分，越大越好', '公式': '0.45*auc_lift + 0.25*W + 0.2*Cliff + 0.1*MI_norm'},
])
show_table('单特征判别指标含义', metric_explain)

### 3.1 BK00 vs NONBK00

BK00 与同流速非断丝样本 `FL00+QJ00` 比较，回答零流速下断丝是否能从背景和敲击中分离。

In [ ]:
cols = ['comparison', 'feature', 'n_positive', 'n_negative', 'auc_abs', 'wasserstein_norm', 'abs_cliff_delta', 'mutual_info', 'discrimination_score']
show_table('3.1 BK00 vs NONBK00 单特征Top20', feature_discrimination[feature_discrimination['comparison'].eq('BK00_NONBK00')][cols], rows=20)

### 3.2 BK00 vs QJ00

只比较零流速断丝与零流速敲击，重点检查抗敲击干扰的断丝特异性。

In [ ]:
cols = ['comparison', 'feature', 'n_positive', 'n_negative', 'auc_abs', 'wasserstein_norm', 'abs_cliff_delta', 'mutual_info', 'discrimination_score']
show_table('3.2 BK00 vs QJ00 单特征Top20', feature_discrimination[feature_discrimination['comparison'].eq('BK00_QJ00')][cols], rows=20)

### 3.3 BK05 vs QJ05

只比较 v0.5 流速断丝与 v0.5 流速敲击，重点检查有流速背景下的抗敲击能力。

In [ ]:
cols = ['comparison', 'feature', 'n_positive', 'n_negative', 'auc_abs', 'wasserstein_norm', 'abs_cliff_delta', 'mutual_info', 'discrimination_score']
show_table('3.3 BK05 vs QJ05 单特征Top20', feature_discrimination[feature_discrimination['comparison'].eq('BK05_QJ05')][cols], rows=20)

### 3.4 BK05 vs NONBK05

BK05 与同流速非断丝样本 `FL05+QJ05` 比较，回答 v0.5 流速下断丝是否可分。

In [ ]:
cols = ['comparison', 'feature', 'n_positive', 'n_negative', 'auc_abs', 'wasserstein_norm', 'abs_cliff_delta', 'mutual_info', 'discrimination_score']
show_table('3.4 BK05 vs NONBK05 单特征Top20', feature_discrimination[feature_discrimination['comparison'].eq('BK05_NONBK05')][cols], rows=20)

### 3.5 BK vs NONBK

合并两种流速后比较全部断丝与全部非断丝，是最终特征排序的主任务。

In [ ]:
cols = ['comparison', 'feature', 'n_positive', 'n_negative', 'auc_abs', 'wasserstein_norm', 'abs_cliff_delta', 'mutual_info', 'discrimination_score']
show_table('3.5 BK vs NONBK 单特征Top20', feature_discrimination[feature_discrimination['comparison'].eq('BK_NONBK')][cols], rows=20)

## 04 特征冗余分析

高相关特征携带的信息高度重复，直接一起进入模型会增加解释成本，也可能放大同一物理量的权重。本章分三部分：4.1 使用 Pearson 相关分析线性冗余；4.2 使用 Spearman 相关分析单调冗余；4.3 对比两种方法，形成“哪些2个或3个/更多特征极高相关、建议保留哪1个”的冗余推荐表。

阈值由 `CONFIG.correlation_threshold` 控制，当前为 0.92。若希望更严格去重可提高到 0.95 或 0.98；若希望更早发现潜在重复表达，可降到 0.85-0.90。

### 4.1 使用 Pearson 相关

Pearson 相关系数度量两个特征的**线性**相关程度：
$$r=\frac{\sum_{i}(x_i-\bar{x})(y_i-\bar{y})}{\sqrt{\sum_{i}(x_i-\bar{x})^2}\sqrt{\sum_{i}(y_i-\bar{y})^2}},\qquad r\in[-1,1]$$

当 $|r|\ge\mathrm{correlation\_threshold}$（当前 0.92）时判定为高相关特征对。它只捕捉线性关系：两个特征只要满足近似比例/仿射关系，即使物理含义不同也会得到很高的 $|r|$，例如包络峰位比 $r_p$ 与包络不对称度 $A_{env}$ 满足 $A_{env}=1-2\,r_p$，理论相关系数恒为 $-1$。

**$|r|=1$ 特征对的甄别**（结合 `fea_cpt_gpu_v2/features.py` 公式）：

- `R_2_1` 与 `R_h`：实现中两者都为 $e_2/(e_1+\varepsilon)$，属于**实现层重复**（一个公式被两个特征名复用）；
- `Ridge_coh` 与 `R_harm`：`Ridge_coh = features["R_harm"]`，是**同一变量赋值两次**；
- `r_p` 与 `A_env`：$A_{env}=1-2r_p$ 是数学恒等，属于**理论上严格线性相反**，不是算错；
- 其余 $|r|=1$ 的对（如部分 `R_wp_*` 小波包节点对）需要结合该频带内信号是否带限、子带能量是否数值相等逐对核实。

In [ ]:
from pccp_feature_mining.feature_selection import build_relevance_series, run_mrmr_ranking, build_final_ranking
from pccp_feature_mining.feature_redundancy import (
    build_correlation_clusters,
    build_redundancy_recommendations,
    compare_correlation_methods,
    compute_correlation_matrix,
    high_correlation_pairs,
)

relevance = build_relevance_series(feature_discrimination, comparison='BK_NONBK')
pearson_corr = compute_correlation_matrix(
    df,
    feature_cols,
    method='pearson',
    max_rows=CONFIG.max_correlation_rows,
    random_state=CONFIG.random_state,
)
pearson_corr.to_csv(RUN_DIR / 'pearson_correlation_matrix.csv', encoding='utf-8-sig')
pearson_high_pairs = high_correlation_pairs(pearson_corr, threshold=CONFIG.correlation_threshold)
write_csv(add_feature_meaning_columns(pearson_high_pairs), RUN_DIR / 'pearson_high_correlation_pairs.csv')
show_table('Pearson高相关特征对Top30', pearson_high_pairs, rows=30)

### 4.2 使用 Spearman 相关

Spearman 相关系数先把特征值转换为秩（rank），再计算秩的 Pearson 相关：
$$\rho = 1-\frac{6\sum_{i}d_i^2}{n(n^2-1)},\qquad d_i=\mathrm{rank}(x_i)-\mathrm{rank}(y_i)$$

它只依赖大小顺序，对尺度拉伸和非线性单调变换更稳健，能发现 Pearson 看不出的**单调**冗余关系。本流程后续 mRMR 的冗余项与最终冗余惩罚均使用 Spearman 相关矩阵。

In [ ]:
spearman_corr = compute_correlation_matrix(
    df,
    feature_cols,
    method='spearman',
    max_rows=CONFIG.max_correlation_rows,
    random_state=CONFIG.random_state,
)
spearman_corr.to_csv(RUN_DIR / 'spearman_correlation_matrix.csv', encoding='utf-8-sig')
spearman_high_pairs = high_correlation_pairs(spearman_corr, threshold=CONFIG.correlation_threshold)
corr = spearman_corr
high_pairs = spearman_high_pairs
correlation_cluster = build_correlation_clusters(spearman_corr, relevance, threshold=CONFIG.correlation_threshold)
write_csv(add_feature_meaning_columns(spearman_high_pairs), RUN_DIR / 'spearman_high_correlation_pairs.csv')
write_csv(add_feature_meaning_columns(high_pairs), RUN_DIR / 'high_correlation_pairs.csv')
write_csv(add_feature_meaning_columns(correlation_cluster), RUN_DIR / 'correlation_cluster.csv')
show_table('Spearman高相关特征对Top30', spearman_high_pairs, rows=30)
show_table('Spearman相关簇Top30', correlation_cluster, rows=30)

### 4.3 Pearson 与 Spearman 结果对比

本节把两种方法命中的高相关特征对（均满足 $|r|\ge0.92$）合并，再按**连通分量**构建冗余组：若特征 a 与 b 高相关、b 与 c 也高相关，则 a、b、c 归入同一组，代表它们共同表达同一类信息。

组内推荐保留 `BK vs NONBK` 判别分最高的 1 个特征作为代表，其余作为冗余删除候选：
$$\mathrm{keep}=\mathop{\arg\max}_{f\in G}\;D_{\mathrm{BK\_NONBK}}(f),\qquad \mathrm{drop}=G\setminus\{\mathrm{keep}\}$$

如果一个组包含 3 个或更多特征，说明同一信息被多个特征重复表达的风险更高，优先合并删除。

In [ ]:
correlation_method_comparison = compare_correlation_methods(pearson_high_pairs, spearman_high_pairs)
redundancy_recommendations = build_redundancy_recommendations(
    pearson_corr,
    spearman_corr,
    relevance=relevance,
    threshold=CONFIG.correlation_threshold,
)
write_csv(add_feature_meaning_columns(correlation_method_comparison), RUN_DIR / 'correlation_method_comparison.csv')
write_csv(add_feature_meaning_columns(redundancy_recommendations), RUN_DIR / 'redundancy_recommendations.csv')
show_table('Pearson与Spearman高相关特征对对比Top40', correlation_method_comparison, rows=40)
show_table('高相关冗余组与保留建议Top40', redundancy_recommendations, rows=40)

## 05 多特征组合搜索

本章分成四个小节，用不同方法构造特征组合并对比。组合评价统一用 LDA 交叉验证：对训练折拟合标准化 + LDA，在验证折计算无方向 AUC 和 LDA 投影分离度。`cv_auc_abs` 越大越好；`lda_separation = |mean(score_pos)-mean(score_neg)| / pooled_std`，越大表示投影后两类分得越开。

### 5.1 mRMR前缀组合

mRMR（Minimum Redundancy Maximum Relevance）是一种贪心特征选择算法，其核心思想是同时最大化特征与标签的**相关性**（relevance），并最小化已选特征之间的**冗余**（redundancy）[Peng et al., 2005]。

**核心公式**

每一步从未选特征 $f$ 中选择以下目标函数最大的特征加入已选集合 $S$：

$$
\mathrm{score}(f) \;=\; D(f) \;-\; \lambda \cdot R(f \mid S)
$$

- $D(f)$：特征 $f$ 与标签的**相关性**分数（单变量判别分）。本程序用 BK vs Non-BK 的 AUC 实现，取 $\max(\mathrm{AUC},\,1-\mathrm{AUC})$，范围 $[0.5, 1]$。
- $R(f\mid S) = \frac{1}{|S|}\sum_{g\in S}\vert\rho(f,g)\vert$：特征 $f$ 与已选集合 $S$ 中各特征的**平均绝对 Spearman 相关系数**，衡量冗余度。
- $\lambda$：冗余惩罚权重，默认 $0.5$（`mrmr_redundancy_weight`）。

**算法流程**

1. 初始化 $S=\emptyset$，候选集为全部特征。
2. 第一轮：所有特征的冗余项为 0，直接选 $D(f)$ 最大的特征加入 $S$。
3. 后续轮次：对每个候选特征 $f$ 计算 $\mathrm{score}(f)$，选取得分最高的 $f^*$ 加入 $S$。
4. 重复直至达到 `mrmr_top_n`（默认 80）个特征后停止。

**后缀组合评估**

对 mRMR 排序结果，取前 $K \in \{5,10,20,30,50\}$ 个特征组成候选子集，用 5 折分层 LDA 交叉验证评估 `cv_auc_abs`（无方向 AUC，越大越好）和 `lda_separation`（LDA 投影后两类分离度 $= \vert\bar{y}_{+} - \bar{y}_{-}\vert / \sigma_{\text{pooled}}$，越大越好）。

**参考文献**

[1] Peng H, Long F, Ding C. Feature selection based on mutual information: criteria of max-dependency, max-relevance, and min-redundancy. *IEEE Trans. PAMI*, 2005, 27(8): 1226-1238.
[2] Ding C, Peng H. Minimum redundancy feature selection from microarray gene expression data. *J. Bioinformatics & Computational Biology*, 2005, 3(2): 185-205.

In [ ]:
from pccp_feature_mining.combination_search import evaluate_mrmr_prefixes, relief_like_ranking, sequential_forward_search

mrmr_rank = run_mrmr_ranking(
    relevance=relevance,
    corr=corr,
    top_n=CONFIG.mrmr_top_n,
    redundancy_weight=CONFIG.mrmr_redundancy_weight,
)
mrmr_prefix = evaluate_mrmr_prefixes(
    df,
    mrmr_rank,
    feature_cols,
    counts=CONFIG.combination_feature_counts,
    random_state=CONFIG.random_state,
)
write_csv(add_feature_meaning_columns(mrmr_rank), RUN_DIR / 'mrmr_rank.csv')
write_csv(add_feature_meaning_columns(mrmr_prefix), RUN_DIR / 'feature_combination_mrmr_prefix.csv')
show_table('mRMR特征排序Top30', mrmr_rank, rows=30)
show_table('mRMR前缀组合评价', mrmr_prefix)

### 5.2 近似 ReliefF 排序

ReliefF 是一种基于**最近邻**的过滤式特征加权算法：对每个样本，考察与其最近的同类别样本（near-hit）和最近的不同类别样本（near-miss）；若某特征能使同类更近、异类更远，则该特征得分上升。

**核心公式**

本实现为工程近似版：先对特征矩阵做 Z-score 标准化（消除量纲影响），再对每个类别 $c\in\{0,1\}$ 的两类贡献累加：
$$W(f)=\sum_{c\in\{0,1\}}\Big[\underbrace{\frac{1}{|\mathcal{X}_c|}\sum_{x_i\in\mathcal{X}_c}\big\lvert x_i^{(f)}-x_{\mathrm{miss}}^{(f)}\big\rvert}_{\text{异类（miss）贡献}}-\underbrace{\frac{1}{|\mathcal{X}_c|}\sum_{x_i\in\mathcal{X}_c}\big\lvert x_i^{(f)}-x_{\mathrm{hit}}^{(f)}\big\rvert}_{\text{同类（hit）贡献}}\Big]$$

其中 $x_i^{(f)}$ 表示样本 $x_i$ 在特征 $f$ 上的取值；$x_{\mathrm{miss}}$ 为与 $x_i$ 最近的异类样本（第 1 近邻），$x_{\mathrm{hit}}$ 为与 $x_i$ 最近的同类样本（取第 2 近邻，排除自身）。

**流程**

1. 构造 BK vs Non-BK 二分类矩阵（`build_binary_frame`，可选抽样至 4000 行）。
2. 用 `StandardScaler` 标准化全部特征。
3. 对每一类样本，用 `NearestNeighbors` 找出 near-hit / near-miss，按上式累加各特征权重。
4. 按 $W(f)$ 降序输出 `relief_score` 作为候选排序。

**与标准 ReliefF 的区别**：标准 ReliefF 随机抽样本并按距离做指数软加权；本实现用全部样本、最近邻硬距离，方向含义一致但数值尺度不同，仅作为与 mRMR 互补的排序视角。

**参考文献**

[1] Kira K, Rendell L A. A practical approach to feature selection[C]. *Proc. 9th Intl. Conf. on Machine Learning (ICML)*, 1992: 249-256.
[2] Robnik-Sikonja M, Kononenko I. Theoretical and empirical analysis of ReliefF and RReliefF[J]. *Machine Learning*, 2003, 55(1): 23-69.

In [ ]:
relieff_rank = relief_like_ranking(df, feature_cols, random_state=CONFIG.random_state)
write_csv(add_feature_meaning_columns(relieff_rank), RUN_DIR / 'relieff_rank.csv')
show_table('近似ReliefF特征排序Top30', relieff_rank, rows=30)

### 5.3 顺序前向选择 SFS

SFS（Sequential Forward Selection）是经典的**包装式**（wrapper）搜索：从空集出发，每轮把"加入后组合判别效果最好"的一个候选特征加入集合，逐步扩张特征子集，直到达到上限或候选耗尽。

**核心公式**

设当前已选集合为 $S_t$、剩余候选为 $C_t$，第 $t$ 轮选择：
$$f^*_t=\mathop{\arg\max}_{f\in C_t}\;\mathrm{score}_{\mathrm{LDA}}\big(S_t\cup\{f\}\big)$$

其中 $\mathrm{score}_{\mathrm{LDA}}(\cdot)$ 为对候选组合做 5 折分层 LDA 交叉验证的无方向 AUC：
$$\mathrm{AUC}_{\mathrm{abs}}=\frac{1}{5}\sum_{v=1}^{5}\max\big(\mathrm{AUC}_v,\;1-\mathrm{AUC}_v\big)$$

**流程**

1. 候选集 = mRMR 前 `sfs_candidate_count`（30）与 ReliefF 前 30 个特征去重后的并集。
2. 初始化 $S=\emptyset$，`best_auc=0.5`。
3. 每轮遍历每个候选 $f$，用 LDA 交叉验证评估 $S\cup\{f\}$，选使 AUC 最大的 $f^*$。
4. 更新 $S\gets S\cup\{f^*\}$、删除该候选，记录 `step / added_feature / cv_auc_abs / best_auc_so_far / lda_separation`。
5. 终止条件：$|S|=\mathrm{sfs\_max\_selected}$（默认 15）或候选为空。

**关键指标**

- `cv_auc_abs`：当前组合交叉验证无方向 AUC，越大越好；
- `lda_separation`：`|mean(score_pos)-mean(score_neg)|/pooled_std`，越大两类投影越分开；
- `best_auc_so_far`：到当前步为止的历史最优 AUC，用于观察是否已收敛。

**参考文献**

[1] Kohavi R, John G H. Wrappers for feature subset selection[J]. *Artificial Intelligence*, 1997, 97(1-2): 273-324.
[2] Pudil P, Novovicova J, Kittler J. Floating search methods in feature selection[J]. *Pattern Recognition Letters*, 1994, 15(11): 1119-1125.

In [ ]:
sfs_candidates = list(dict.fromkeys(mrmr_rank['feature'].head(CONFIG.sfs_candidate_count).tolist() + relieff_rank['feature'].head(CONFIG.sfs_candidate_count).tolist()))
sfs_selection_path = sequential_forward_search(
    df,
    sfs_candidates,
    feature_cols,
    max_selected=CONFIG.sfs_max_selected,
    random_state=CONFIG.random_state,
)
write_csv(add_feature_meaning_columns(sfs_selection_path), RUN_DIR / 'sfs_selection_path.csv')
show_table('SFS逐步选择路径', sfs_selection_path)

### 5.4 多方法结果对比

本节把 mRMR 前缀组合与 SFS 路径合并成统一对比表。优先关注 `cv_auc_abs` 高、`lda_separation` 高且特征数量较少的组合。

In [ ]:
if sfs_selection_path.empty:
    feature_combination_search = mrmr_prefix.copy()
else:
    feature_combination_search = pd.concat([
        mrmr_prefix,
        sfs_selection_path.rename(columns={'step': 'sfs_step'})[['method', 'comparison', 'feature_count', 'cv_auc_abs', 'lda_separation', 'selected_features']]
    ], ignore_index=True, sort=False)

write_csv(add_feature_meaning_columns(feature_combination_search), RUN_DIR / 'feature_combination_search.csv')
show_table('多特征组合搜索方法对比Top20', feature_combination_search.sort_values('cv_auc_abs', ascending=False), rows=20)

## 06 特征稳定性分析

本节实现 **Bootstrap 稳定性分析**：重复构造多个"全量 BK + 等量 Other"的平衡子样本，在每个子样本上重新计算全部特征的单特征 AUC 排名。若某特征在多轮抽样中都进入 Top-K，说明其判别力不是由某次抽样偶然产生。

**核心公式**

设共 $B$ 轮（`bootstrap_rounds`，默认 200），特征总数 $m$。

- **第 $b$ 轮样本构成**（平衡抽样）：
  $$S^{(b)}=S_{\mathrm{BK}}\;\cup\;S_{\mathrm{Other}}^{(b)},\qquad \big|S_{\mathrm{Other}}^{(b)}\big|=\big|S_{\mathrm{BK}}\big|$$
  $S_{\mathrm{BK}}$ 为全部断丝样本；$S_{\mathrm{Other}}^{(b)}$ 从非断丝样本中按 `sampling_group_id` 分组抽取等量样本：QJ 事件保持整组完整，FL 窗口按独立窗口处理。

- **第 $b$ 轮特征分数**（无方向 AUC lift）：
  $$\mathrm{lift}_b(f)=\max\big(\mathrm{AUC}_b(f),\;1-\mathrm{AUC}_b(f)\big)-0.5,\qquad f=1,\dots,m$$
  再按 $\mathrm{lift}_b(f)$ 降序得到本轮名次 $\mathrm{rank}_b(f)\in[1,m]$。

- **汇总统计**（$B$ 轮平均）：
  $$\overline{\mathrm{rank}}(f)=\frac{1}{B}\sum_{b=1}^{B}\mathrm{rank}_b(f),\qquad
  \hat\sigma_{\mathrm{rank}}(f)=\sqrt{\frac{1}{B}\sum_{b=1}^{B}\Big(\mathrm{rank}_b(f)-\overline{\mathrm{rank}}(f)\Big)^2}
  $$
  $$\mathrm{freq}_K(f)=\frac{1}{B}\sum_{b=1}^{B}\mathbb{1}\big[\mathrm{rank}_b(f)\le K\big],\qquad K\in\{10,20,30\}$$

**关键指标含义**

| 指标 | 公式 | 含义 |
|---|---|---|
| `mean_rank` | $\overline{\mathrm{rank}}(f)$ | 平均名次，越小越靠前 |
| `rank_std` | $\hat\sigma_{\mathrm{rank}}(f)$ | 名次标准差，越小越稳定 |
| `topK_frequency` | $\mathrm{freq}_K(f)$ | 进入 Top-K 的轮次占比，越高越稳定 |
| `mean_auc_lift` | $\frac{1}{B}\sum_b\mathrm{lift}_b(f)$ | 平均判别力 |
| `rank_stability_score` | $1-\frac{\overline{\mathrm{rank}}(f)-1}{m}$ | 归一化稳定性分，用于最终排序加权 |

结果按 `top20_frequency`、`mean_auc_lift`、`mean_rank` 排序输出。

**参数建议**：快速调试可把 `bootstrap_rounds` 设为 20-50，正式报告建议 200-1000；BK 样本增加后 Other 仍建议与 BK 等量抽样；保守场景可把 `top_ks` 扩展为 `(10, 20, 30, 50)`。

**参考文献**

[1] Efron B. Bootstrap methods: another look at the jackknife[J]. *The Annals of Statistics*, 1979, 7(1): 1-26.
[2] Efron B, Tibshirani R J. An Introduction to the Bootstrap[M]. New York: Chapman & Hall, 1993.

In [ ]:
from pccp_feature_mining.bootstrap_stability import run_bootstrap_stability

bootstrap_parameter_table = pd.DataFrame([
    {'参数': 'rounds', '当前值': CONFIG.bootstrap_rounds, '计算/含义': '重复抽样轮数'},
    {'参数': 'BK rows per round', '当前值': int(df['target_label'].eq('BK').sum()), '计算/含义': '每轮保留全部BK样本'},
    {'参数': 'Other rows per round', '当前值': int(df['target_label'].eq('BK').sum()), '计算/含义': '每轮从Other抽取与BK等量的样本'},
    {'参数': 'top_ks', '当前值': str(CONFIG.bootstrap_top_ks), '计算/含义': '统计进入Top-K集合的频率'},
    {'参数': 'random_state', '当前值': CONFIG.random_state, '计算/含义': '随机数种子，保证结果可复现'},
])
show_table('Bootstrap稳定性分析参数', bootstrap_parameter_table)

feature_stability = run_bootstrap_stability(
    df,
    feature_cols,
    rounds=CONFIG.bootstrap_rounds,
    top_ks=CONFIG.bootstrap_top_ks,
    random_state=CONFIG.random_state,
)
write_csv(add_feature_meaning_columns(feature_stability), RUN_DIR / 'feature_stability.csv')
show_table('Bootstrap稳定特征Top40', feature_stability, rows=40)

## 07 跨流速一致性分析

本节比较 `BK00 vs FL00` 和 `BK05 vs FL05`，寻找在 v0 与 v0.5 流速下都能稳定区分断丝与流噪背景的特征。表中列举的是按 `cross_flow_score` 排序后的前若干特征，因为这些特征同时满足“两个流速下都有判别力、判别方向一致、受流速本身影响较小”。

主要公式：`auc_lift = max(AUC, 1-AUC) - 0.5`；`cross_condition_consistency = min(lift00, lift05) / max(lift00, lift05)`；`median_shift_norm = |median(a)-median(b)| / scale`，其中 `scale` 优先用两组数据合并后的 IQR；`cross_flow_score = direction_same * (0.55*(lift00+lift05)+0.45*consistency) / (1+0.5*bk_shift+0.25*fl_shift)`。

In [ ]:
from pccp_feature_mining.cross_flow_analysis import evaluate_cross_flow_features

cross_flow_metric_explain = pd.DataFrame([
    {'列名': 'auc_BK00_vs_FL00', '含义': 'v0流速下BK相对FL的原始AUC', '方向': '偏离0.5越大越好', '计算方式': 'roc_auc_score(BK00=1, FL00=0)'},
    {'列名': 'auc_BK05_vs_FL05', '含义': 'v0.5流速下BK相对FL的原始AUC', '方向': '偏离0.5越大越好', '计算方式': 'roc_auc_score(BK05=1, FL05=0)'},
    {'列名': 'auc_lift_BK00_vs_FL00', '含义': 'v0无方向AUC提升', '方向': '越大越好', '计算方式': 'max(AUC, 1-AUC)-0.5'},
    {'列名': 'auc_lift_BK05_vs_FL05', '含义': 'v0.5无方向AUC提升', '方向': '越大越好', '计算方式': 'max(AUC, 1-AUC)-0.5'},
    {'列名': 'direction_same', '含义': '两个流速下BK方向是否一致', '方向': '1优于0', '计算方式': '(auc00>=0.5)==(auc05>=0.5)'},
    {'列名': 'cross_condition_consistency', '含义': '两个流速下判别强度是否接近', '方向': '越大越好', '计算方式': 'min(lift00,lift05)/max(lift00,lift05)'},
    {'列名': 'bk00_bk05_median_shift_norm', '含义': 'BK特征值受流速变化影响的归一化中位数位移', '方向': '越小越好', '计算方式': '|median(BK00)-median(BK05)|/scale'},
    {'列名': 'fl00_fl05_median_shift_norm', '含义': 'FL背景特征值受流速变化影响的归一化中位数位移', '方向': '越小越好', '计算方式': '|median(FL00)-median(FL05)|/scale'},
    {'列名': 'flow_sensitivity_penalty', '含义': '流速敏感性惩罚项', '方向': '越小越好', '计算方式': 'bk_shift + 0.5*fl_shift'},
    {'列名': 'cross_flow_score', '含义': '跨流速稳定断丝判别综合分', '方向': '越大越好', '计算方式': '方向一致性、双流速AUC提升、一致性和流速惩罚的综合'},
])
show_table('跨流速一致性指标含义与公式', cross_flow_metric_explain)

cross_flow_feature = evaluate_cross_flow_features(df, feature_cols)
write_csv(add_feature_meaning_columns(cross_flow_feature), RUN_DIR / 'cross_flow_feature.csv')
show_table('跨流速一致性特征Top40', cross_flow_feature, rows=40)

## 08 最终特征评价与结论

本节把前面各阶段分数融合为最终排序：判别能力、抗 QJ 干扰、Bootstrap 稳定性、跨流速一致性和冗余惩罚。

**核心公式**

$$\mathrm{final\_score}=\frac{0.42\,s_{\mathrm{bk}}+0.23\,s_{\mathrm{qj}}+0.20\,s_{\mathrm{stab}}+0.15\,s_{\mathrm{cf}}}{1+0.65\,\mathrm{flow\_penalty}+0.45\,\mathrm{red\_penalty}}$$

各符号含义：

| 符号 | 来源 | 含义 |
|---|---|---|
| $s_{\mathrm{bk}}$ | 表 3.5 `BK_NONBK` 判别分 | 区分断丝与其他样本的能力 |
| $s_{\mathrm{qj}}$ | `BK_QJ` 判别分 | 区分断丝与敲击干扰的能力 |
| $s_{\mathrm{stab}}$ | Bootstrap 稳定性分 $=0.6\,\mathrm{top20\_freq}+0.4\,\mathrm{rank\_stability\_score}$ | 排名跨抽样稳定性 |
| $s_{\mathrm{cf}}$ | 跨流速 `cross_flow_score` | 两种流速下均稳定判别 |
| $\mathrm{flow\_penalty}$ | 跨流速敏感性 | BK00 与 BK05 中位数位移，工况敏感惩罚 |
| $\mathrm{red\_penalty}$ | 冗余惩罚 `redundancy_penalty` | 与更高判别分特征的最大绝对 Spearman 相关系数 |

**分级规则**：
- `feature_grade = A`：$s\geq$ 上四分位数 且 冗余惩罚 < 0.92 且 稳定性分 > 0 → 稳定断丝核心候选；
- `B`：$s\geq$ 0.45 分位 或 排名前 80 → 候选特征；
- `C`：其余 → 工况相关、冗余较高或稳定性不足。

In [ ]:
final_feature_ranking = build_final_ranking(
    feature_columns=feature_cols,
    discrimination=feature_discrimination,
    stability=feature_stability,
    cross_flow=cross_flow_feature,
    redundancy=correlation_cluster,
    mrmr_rank=mrmr_rank,
)
write_csv(add_feature_meaning_columns(final_feature_ranking), RUN_DIR / 'final_feature_ranking.csv')

final_ranking_path = RUN_DIR / 'plots/final_feature_ranking_top30.png'
top_boxplot_path = RUN_DIR / 'plots/top_feature_boxplots.png'
plot_final_ranking(final_feature_ranking, final_ranking_path, top_n=30)
plot_top_feature_boxplots(
    df,
    final_feature_ranking.head(CONFIG.max_plot_features)['feature'].tolist(),
    top_boxplot_path,
    max_features=12,
)

show_table('最终特征排序Top60', final_feature_ranking, rows=60)
show_image('最终特征Top30排序图', final_ranking_path)
show_image('最终Top特征六类箱线图', top_boxplot_path)

## 09 特征挖掘后的分类测试

本节验证挖掘出的特征能否支持实际分类，而不只是排名。

**模型与数据集划分**

- 模型：逻辑回归（Logistic Regression）、线性 SVM、RBF-SVM，均以标准化 + 该模型组成 pipeline。
- 划分规则更严格：在每个来源标签内部按 `source_file_name` 分组划分训练/测试，同一源文件不跨集合；训练集多数类最多采样到 `classification_max_train_rows_per_class`（3000），测试集保持分组划分后的真实分布，避免同源信息泄漏与样本混入。

**评价指标公式**（二分类，TP/FP/TN/FN 为四类计数）

- 平衡准确率：$\mathrm{balanced\_acc}=\frac{1}{2}\big(\frac{TP}{TP+FN}+\frac{TN}{TN+FP}\big)$，适合不均衡数据；
- AUC：ROC 曲线下面积，衡量排序能力；
- 正类召回率（断丝漏检程度）：$\mathrm{recall\_positive}=\frac{TP}{TP+FN}$；
- 特异度（误报程度）：$\mathrm{specificity}=\frac{TN}{TN+FP}$。

综合选择"综合分最高且特征数较少"的推荐组合。

In [ ]:
from pccp_feature_mining.classification_test import run_classification_tests

classification_results, comparison_feature_recommendations = run_classification_tests(
    df,
    feature_discrimination=feature_discrimination,
    final_ranking=final_feature_ranking,
    output_dir=RUN_DIR / 'classification',
    feature_counts=CONFIG.classification_feature_counts,
    max_train_rows_per_class=CONFIG.classification_max_train_rows_per_class,
    random_state=CONFIG.random_state,
)
write_csv(add_feature_meaning_columns(classification_results), RUN_DIR / 'classification_test_results.csv')
write_csv(add_feature_meaning_columns(comparison_feature_recommendations), RUN_DIR / 'comparison_feature_recommendations.csv')
write_markdown_summary(RUN_DIR / 'summary_report.md', dataset_summary, final_feature_ranking, dataset.load_report)

show_table('分类测试结果Top50', classification_results, rows=50)
show_table('五个比较任务的推荐特征组合', comparison_feature_recommendations)

In [ ]:
conclusion_rows = []
for _, row in comparison_feature_recommendations.iterrows():
    feats = str(row['recommended_features']).split(';')
    conclusion_rows.append({
        '比较任务': row['comparison'],
        '建议特征数量': int(row['recommended_feature_count']),
        '建议模型': row['recommended_model'],
        'balanced_accuracy越大越好': row['balanced_accuracy'],
        'auc越大越好': row['auc'],
        '断丝召回率越大越好': row['recall_positive'],
        'Other/QJ特异度越大越好': row['specificity'],
        '具体特征': '；'.join(feats),
        '具体特征中文含义': describe_feature_list(feats),
    })
conclusion_table = pd.DataFrame(conclusion_rows)
write_csv(conclusion_table, RUN_DIR / 'classification_conclusion_table.csv')
show_table('分类验证结论汇总', conclusion_table)

## 10 一键运行入口

本节提供命令式入口，适合不逐节查看中间结果时直接生成全部产物。它会自动创建带时间戳的输出目录，并生成方案约定的核心 CSV、分布图、组合搜索结果、最终排序和分类测试结果。正式分析建议保持 `bootstrap_rounds=200` 或更高；快速调试可设置 `max_rows_per_label` 和较小的 `bootstrap_rounds`。

In [ ]:
from pccp_feature_mining.run_all import run_pccp_feature_mining

# summary = run_pccp_feature_mining(CONFIG)
# summary